# Module 13 Assignment

## _*Make sure to read the instruction document before beginning the assignment_

In [0]:
# Import libraries

import requests
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, date_format, current_date, when, count, countDistinct, avg, sum
from datetime import datetime

# Initialize Spark session
spark = SparkSession.builder.appName("FoursquareDataEnhanced").getOrCreate()

### Question #1 - Initialize Foursquare API (0 points)

In [0]:
# Foursquare API details

api_url = "https://api.foursquare.com/v3/places/search"
headers = {
    "Accept": "application/json",
    "Authorization": "fsq3J/RnWZHJd5/4Vt00u76qFZKFTUWIyUVcu6hT88/q6oc=" 
}


### Question #2 - Retrieve 1000 Rows of Data (Restaurants) from Foursquare Places in Baltimore (20 points)

In [0]:
# Define search parameters
# Search for restaurants in Baltimore, Maryland
# This will require investigating the data to discover how the fields appear.

# The API is limited to 20 results per request
# If you have limit=20, that means you're getting results 1-20
# Give it offset=21 and you'll get 21-40.

def fetch_places_data(limit=20, max_results=1000, query='restaurant', near='New York, NY'):
    collected_data = []
    offset = 1  # Start from 1 as per API's paging behavior

    while len(collected_data) < max_results:
        params = {
            'query': query,
            'near': near,
            'limit': limit,
            'offset': offset
        }

        response = requests.get(api_url, headers=headers, params=params)
        if response.status_code != 200:
            print(f"Error: {response.status_code}, {response.text}")
            break

        data = response.json().get('results', [])

        if not data:
            break

        for place in data:
            location = place.get('location', {})
            categories = place.get('categories', [{}])

            collected_data.append({
                'fsq_id': place.get('fsq_id'),
                'name': place.get('name'),
                'latitude': location.get('lat'),
                'longitude': location.get('lng'),
                'distance': place.get('distance'),
                'category_id': categories[0].get('id') if categories else None,
                'category_name': categories[0].get('name') if categories else None,
                'region': location.get('region'),
                'locality': location.get('locality'),
                'postcode': location.get('postcode'),
                'address': location.get('address'),
                'timezone': location.get('timezone')
            })

        offset += limit

    return collected_data[:max_results]

In [0]:
# Create a loop to retrieve the restaurant data by iteratively updating the 'offset' parameter
# Set the maximum number of results to retrieve to 1000

places_data = fetch_places_data(limit=20, max_results=1000, query='restaurant', near='New York, NY')

In [0]:
places = [
    {
        'fsq_id': str(p.get('fsq_id', '')),  
        'name': str(p.get('name', '')),  
        'latitude': float(p.get('latitude', 0.0)) if p.get('latitude') is not None else 0.0,  
        'longitude': float(p.get('longitude', 0.0)) if p.get('longitude') is not None else 0.0, 
        'distance': float(p.get('distance', 0.0)) if p.get('distance') is not None else 0.0,  
        'category_id': str(p.get('category_id', '')),  
        'category_name': str(p.get('category_name', '')),  
        'region': str(p.get('region', '')),  
        'locality': str(p.get('locality', '')),  
        'postcode': str(p.get('postcode', '')),  
        'address': str(p.get('address', '')),  
        'timezone': str(p.get('timezone', '')) if p.get('timezone') else '' 
    }
    for p in places_data if p.get('fsq_id') 
]


### Question #3 - Convert to PySpark DataFrame (10 points)

In [0]:
# Convert Python dictionary to PySpark DataFrame
df = spark.createDataFrame(places)

In [0]:
# Add a column with today's date (YYYYMMDD) as 'load_dt'
df = df.withColumn("load_dt", date_format(current_date(), "yyyyMMdd"))

In [0]:
# Display() the first 5 records of the PySpark DataFrame
df.show(5, truncate=False)

+------------+-----------+-----------------+--------+------------------------+--------+--------+---------+---------------+--------+------+--------+--------+
|address     |category_id|category_name    |distance|fsq_id                  |latitude|locality|longitude|name           |postcode|region|timezone|load_dt |
+------------+-----------+-----------------+--------+------------------------+--------+--------+---------+---------------+--------+------+--------+--------+
|5 E 19th St |13352      |Thai Restaurant  |380.0   |57e83df3498eebbe238cb36f|0.0     |New York|0.0      |Thai Villa     |10003   |NY    |        |20250424|
|8 E 18th St |13332      |Salad Restaurant |472.0   |5673b4b5498ed6368456b30d|0.0     |New York|0.0      |Sweetgreen     |10003   |NY    |        |20250424|
|36 E 22nd St|13289      |Korean Restaurant|510.0   |5dcdce77738bac0008d93972|0.0     |New York|0.0      |Jua            |10010   |NY    |        |20250424|
|42 E 20th St|13003      |Bar              |546.0   |3fd66

### Question #4 - Create Bronze Table (20 points)

In [0]:
# Check if the database exists
databases = spark.sql("SHOW DATABASES").collect()
db_exists = any(db['databaseName'] == 'places_db' for db in databases)

# If it doesn't exist, create it; if it exists, drop and recreate
if not db_exists:
    print("Creating new database 'places_db'.")
    spark.sql("CREATE DATABASE places_db")
else:
    print("Database 'places_db' already exists.")

# Switch to the places_db database
try:
    spark.sql("USE places_db")
    print("Switched to 'places_db'.")
except Exception as e:
    print(f"Error switching to 'places_db': {str(e)}")


Creating new database 'places_db'.
Switched to 'places_db'.


In [0]:
# Write a Bronze table with the raw data to the places_db database with the 'delta' format (overwrite if it already exists)
# The table should be named <your_last_name>_bronze, i.e. 'mosko_bronze'

table_name = "hito_bronze"

# Write the raw places data into the Bronze table in Delta format
df.write.format("delta").mode("overwrite").saveAsTable(table_name)


In [0]:
# Query all records of the bronze table you created above
bronze_table = spark.sql("SELECT * FROM hito_bronze")

# display() the first 5 records
bronze_table.show(5)

+------------+-----------+-----------------+--------+--------------------+--------+--------+---------+---------------+--------+------+--------+--------+
|     address|category_id|    category_name|distance|              fsq_id|latitude|locality|longitude|           name|postcode|region|timezone| load_dt|
+------------+-----------+-----------------+--------+--------------------+--------+--------+---------+---------------+--------+------+--------+--------+
| 5 E 19th St|      13352|  Thai Restaurant|   380.0|57e83df3498eebbe2...|     0.0|New York|      0.0|     Thai Villa|   10003|    NY|        |20250424|
| 8 E 18th St|      13332| Salad Restaurant|   472.0|5673b4b5498ed6368...|     0.0|New York|      0.0|     Sweetgreen|   10003|    NY|        |20250424|
|36 E 22nd St|      13289|Korean Restaurant|   510.0|5dcdce77738bac000...|     0.0|New York|      0.0|            Jua|   10010|    NY|        |20250424|
|42 E 20th St|      13003|              Bar|   546.0|3fd66200f964a520a...|     0.0

### Question #5 - Create Silver Table (20 points)

In [0]:
# Clean the raw Bronze table following the steps below:

# Keep the following columns: 
# "fsq_id", "name", "distance", "category_name", "region", "locality", "postcode", "address"
silver_df = bronze_table.select("fsq_id", "name", "distance", "category_name", "region", "locality", "postcode", "address")

# Remove duplicate rows from Silver DataFrame
silver_df = silver_df.dropDuplicates()

In [0]:
# Write a Silver table to the places_db database with the 'delta' format (overwrite if it already exists)
# The table should be named <your_last_name>_silver, i.e. 'mosko_silver'
silver_df.write.format("delta").mode("overwrite").saveAsTable("hito_silver")

In [0]:
# Query all records of the silver table you created above
silver_table = spark.sql("SELECT * FROM hito_silver")

# display() the first 5 records
silver_table.show(5)

+--------------------+-------------------+--------+-------------------+------+--------+--------+----------------+
|              fsq_id|               name|distance|      category_name|region|locality|postcode|         address|
+--------------------+-------------------+--------+-------------------+------+--------+--------+----------------+
|5e55a57d55d910000...|            Barbuto|  1453.0| Italian Restaurant|    NY|New York|   10014|  113 Horatio St|
|585164b77220e6221...|4 Charles Prime Rib|  1028.0|American Restaurant|    NY|New York|   10014|    4 Charles St|
|568c0ce238fafac5f...| Mah-Ze-Dahr Bakery|  1005.0|             Bakery|    NY|New York|   10011|28 Greenwich Ave|
|5ee66a36459579000...|             Soothr|  1187.0|       Cocktail Bar|    NY|New York|   10003|   204 E 13th St|
|5673b4b5498ed6368...|         Sweetgreen|   472.0|   Salad Restaurant|    NY|New York|   10003|     8 E 18th St|
+--------------------+-------------------+--------+-------------------+------+--------+-

### Question #6 - Create Gold Table (20 points)

In [0]:
# Write a PySpark query to group the Silver DataFrame (silver_df) by three columns:
#   category_name: The name of the category (e.g., "Restaurant").
#   region: The state or region of the place.
#   locality: The city or locality of the place.
#
# Calculate the following aggregations for each group:
#   num_places: The total number of places in each group (count of fsq_id).
#   avg_distance: The average distance of the places from the query location.
#   unique_postcodes: The number of unique postcodes in each group.
#
# Store the aggregated results in a new DataFrame named gold_df

gold_df = silver_df.groupBy("category_name", "region", "locality") \
    .agg(
        count("fsq_id").alias("num_places"),  
        avg("distance").alias("avg_distance"), 
        countDistinct("postcode").alias("unique_postcodes") 
    )

In [0]:
# Write a Gold table to the places_db database with the 'delta' format (overwrite if it already exists)
# The table should be named <your_last_name>_gold, i.e. 'mosko_gold'
gold_df.write.format("delta").mode("overwrite").saveAsTable("hito_gold")

## Question #7 - Query the Gold Table (10 points)

In [0]:
# Query the gold table and store results in a PySpark DataFrame with an avg_distance of less than 3000
query = """
SELECT * 
FROM hito_gold
WHERE avg_distance < 3000
"""
gold_table = spark.sql(query)

# display() all records
gold_table.show()

+--------------------+------+--------+----------+------------------+----------------+
|       category_name|region|locality|num_places|      avg_distance|unique_postcodes|
+--------------------+------+--------+----------+------------------+----------------+
|    Salad Restaurant|    NY|New York|         1|             472.0|               1|
|   Korean Restaurant|    NY|New York|         1|             510.0|               1|
|  Mexican Restaurant|    NY|New York|         1|            1118.0|               1|
| American Restaurant|    NY|New York|         1|            1028.0|               1|
|          Bagel Shop|    NY|New York|         1|            1253.0|               1|
|  Italian Restaurant|    NY|New York|         3|1408.3333333333333|               1|
|     Thai Restaurant|    NY|New York|         1|             380.0|               1|
|              Bakery|    NY|New York|         3| 756.3333333333334|               2|
|   French Restaurant|    NY|New York|         3|1017.

## Submitting the Assignment
Submit your Jupyter Notebook file with **all cells executed and outputs displayed**.

Export the notebook from Databricks as an html file (File > Export > HTML) and upload to Canvas: \<student-first-intial>\<student-last-name>\-module13.html
    
Upload a Jupyter Notebook with your code: \<student-first-intial>\<student-last-name>\-module13.ipyn